In [ ]:
# !pip install bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !pip install datasets

In [ ]:
import pandas as pd

In [ ]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')
# token

In [ ]:
df = pd.read_csv("/content/pre-processed_reasoning.csv")
df = df[4:]
df = df.reset_index(drop=True)
df

,input,output
0,Dice is the leading career destination for tec...,"[{""thinking"": {""step_1"": {""title"": ""Understand..."
1,"In a world of possibilities, pursue one with e...","[{""thinking"": {""step_1"": {""title"": ""Understand..."
2,About the job\nAbout Rocket Lawyer\n\nWe belie...,"[{""thinking"": {""step_1"": {""title"": ""This role ..."
3,About the job\nThis role is with Maximus. WayU...,"[{""thinking"": {""step_1"": {""title"": ""Understand..."
4,About the job\nXometry (NASDAQ: XMTR) powers t...,"[{""thinking"": {""step_1"": {""title"": ""Understand..."
...,...,...
1001,Dimensional is a privately owned global invest...,"[{""thinking"": {""step_1"": {""title"": ""Understand..."
1002,Sr OR & Advanced Analytics Specialist I (Data ...,"[{""thinking"": {""step_1"": {""title"": ""Understand..."
1003,"**Portfolio Manager, Information Capital and D...","[{""thinking"": {""step_1"": {""title"": ""Understand..."
1004,"As a Data Architect, your role will be to tran...","[{""thinking"": {""step_1"": {""title"": ""Understand..."


In [ ]:
import pandas as pd
import json

formatted_data = []

def format_output(skills):
    output_lines = []
    for item in skills:
        output_lines.append(f"- Skill: {item['skill']}\n  Reason: {item['reason']}")
    return "\n".join(output_lines)

for index, row in df.iterrows():
    try:
        job_description = row["input"]
        output_str = row["output"]
        if pd.isna(output_str):
            raise ValueError("Empty output")

        record_list = json.loads(output_str)
        record = record_list[0]

        thinking = record.get("thinking", {})
        step_1 = thinking.get("step_1", {})
        step_2 = thinking.get("step_2", {})
        step_3 = thinking.get("step_3", {})

        # Include full step-by-step reasoning in the prompt
        thinking_str = json.dumps(thinking, indent=2)

        # Construct skills
        explicit_skills = [{"skill": k, "reason": v} for k, v in step_2.items()]
        implicit_skills = [{"skill": k, "reason": v} for k, v in step_3.items()]
        all_skills = explicit_skills + implicit_skills

        prompt = (
            "You are an AI assistant that extracts skills from job descriptions using chain-of-thought reasoning.\n"
            "Think step-by-step and provide both the skills and your reasoning.\n\n"
            f"Job Description:\n{job_description}\n\n"
            f"Thinking:\n{thinking_str}\n\n"
            "Answer:"
        )

        completion = format_output(all_skills).strip()

        formatted_data.append({
            "prompt": prompt,
            "completion": completion
        })

    except (json.JSONDecodeError, TypeError, ValueError) as e:
        print(f"Row {index} caused error: {e}")
        print(f"Raw output:\n{row['output']}\n{'='*60}")
        continue

# Save to JSONL
with open("formatted_data.jsonl", "w", encoding="utf-8") as f:
    for item in formatted_data:
        f.write(json.dumps(item) + "\n")

print("JSONL conversion complete.")


JSONL conversion complete.
